# Machine Learning Pipeline: Belgian Motor Third-Party Liability Insurance Claims Prediction

## Project Overview

This notebook builds a **machine learning model to predict insurance claim severity** (the financial amount paid out) for Belgian motor third-party liability insurance claims.

### Why This Matters
Insurance companies need to estimate claim payouts to set appropriate premiums and manage risk. By predicting claim values based on policyholder and vehicle characteristics, insurers can:
- Price policies more accurately
- Identify high-risk factors
- Allocate resources more efficiently

### What We'll Do
1. **Load and validate data** - Understand the raw dataset
2. **Clean and prepare data** - Handle missing values and remove problematic features
3. **Engineer features** - Select the best variables for prediction
4. **Build preprocessing pipelines** - Standardize numeric features and encode categorical ones
5. **Train 7 different models** - Compare multiple machine learning approaches
6. **Track experiments with MLflow** - Keep detailed records of each model's performance
7. **Save and deploy the best model** - Export it for production use

### Dataset: beMTPL16
This Belgian motor third-party liability dataset contains **70,791 insurance claims** with 19 variables describing the policyholder, vehicle, and claim circumstances.

## Quick Start

Run all cells in order. Each cell depends on the previous step.

**View Results**: After training completes, section 5 shows model comparison. Run `mlflow ui` to view the experiment dashboard at http://localhost:5000

**Optional Cloud Upload**: Section 7 uploads the best model to Hugging Face Hub for deployment.

## Data Loading

The dataset is stored as an **R data file (.rda)**, which is a compressed format commonly used in the R programming language. We need special tools to read it in Python.

**What happens in the code below:**
1. Install `pyreadr` - a library that can read R files
2. Download the beMTPL16 dataset from the CAS (Casualty Actuarial Society) datasets repository
3. Cache the file locally so we don't re-download it on every run
4. Convert it to a Pandas DataFrame so we can work with it in Python
5. Display the first few rows to verify it loaded correctly

In [1]:
%pip install pyreadr huggingface_hub -q
import requests
import os
import pyreadr
import pandas as pd
import numpy as np

# Data is stored in data/ directory for cleaner project structure
DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)

raw_url = 'https://github.com/dutangc/CASdatasets/raw/master/data/beMTPL16.rda'
file_name = os.path.join(DATA_DIR, os.path.basename(raw_url))
print(f"Downloading dataset to {file_name}...")

try:
    if not os.path.exists(file_name):
        response = requests.get(raw_url)
        response.raise_for_status()
        with open(file_name, 'wb') as f:
            f.write(response.content)
        print("Downloaded successfully")
    else:
        print("Using cached file")

    result = pyreadr.read_r(file_name)
    df_beMTPL16 = result['beMTPL16']
    print(f"Data loaded: {df_beMTPL16.shape[0]} rows, {df_beMTPL16.shape[1]} columns")
except Exception as e:
    print(f"Error: {e}")

Note: you may need to restart the kernel to use updated packages.
Using cached file
Data loaded: 70791 rows, 19 columns


## Pipeline Configuration

All configurable parameters are loaded from the `.env` file in the project root. This allows you to modify settings without changing the notebook code.

**To change configuration:**
1. Open the `.env` file in the project root
2. Modify the values as needed
3. Save the file
4. Re-run this notebook from the beginning

**Available configuration options:**
- `CLEAN_MLFLOW_RUNS` - Set to `True` to delete all MLflow history before training
- `MLFLOW_EXPERIMENT_NAME` - Name for the MLflow experiment
- `TEST_SIZE` - Fraction of data for testing (default: 0.2)
- `RANDOM_STATE` - Random seed for reproducibility (default: 42)
- Model hyperparameters for XGBoost, Random Forest, Ridge, Lasso
- `UPLOAD_TO_HUB` - Set to `True` to upload best model to Hugging Face

In [2]:
%pip install python-dotenv -q
import os
from dotenv import load_dotenv

# Load configuration from .env file
load_dotenv(override=True)

# Helper function to parse boolean values from environment variables
def get_env_bool(key, default=False):
    """Convert string environment variable to boolean."""
    value = os.getenv(key, str(default)).lower()
    return value in ('true', '1', 'yes', 'on')

def get_env_float(key, default):
    """Get float value from environment variable."""
    return float(os.getenv(key, default))

def get_env_int(key, default):
    """Get integer value from environment variable."""
    return int(os.getenv(key, default))

# Load all configuration values
CONFIG = {
    # MLflow settings
    'CLEAN_MLFLOW_RUNS': get_env_bool('CLEAN_MLFLOW_RUNS', False),
    'MLFLOW_EXPERIMENT_NAME': os.getenv('MLFLOW_EXPERIMENT_NAME', 'Belgian_MTPL_Severity'),
    
    # Data split settings
    'TEST_SIZE': get_env_float('TEST_SIZE', 0.2),
    'RANDOM_STATE': get_env_int('RANDOM_STATE', 42),
    
    # XGBoost Basic settings
    'XGBOOST_N_ESTIMATORS': get_env_int('XGBOOST_N_ESTIMATORS', 1000),
    'XGBOOST_LEARNING_RATE': get_env_float('XGBOOST_LEARNING_RATE', 0.05),
    'XGBOOST_MAX_DEPTH': get_env_int('XGBOOST_MAX_DEPTH', 5),
    'XGBOOST_EARLY_STOPPING_ROUNDS': get_env_int('XGBOOST_EARLY_STOPPING_ROUNDS', 50),
    
    # Random Search settings
    'RANDOM_SEARCH_N_ITER': get_env_int('RANDOM_SEARCH_N_ITER', 50),
    'RANDOM_SEARCH_CV_FOLDS': get_env_int('RANDOM_SEARCH_CV_FOLDS', 5),
    
    # Actuarial XGBoost settings
    'ACTUARIAL_N_ESTIMATORS': get_env_int('ACTUARIAL_N_ESTIMATORS', 500),
    'ACTUARIAL_LEARNING_RATE': get_env_float('ACTUARIAL_LEARNING_RATE', 0.1),
    'ACTUARIAL_MAX_DEPTH': get_env_int('ACTUARIAL_MAX_DEPTH', 6),
    'ACTUARIAL_SUBSAMPLE': get_env_float('ACTUARIAL_SUBSAMPLE', 0.8),
    'ACTUARIAL_COLSAMPLE_BYTREE': get_env_float('ACTUARIAL_COLSAMPLE_BYTREE', 0.8),
    'ACTUARIAL_EARLY_STOPPING_ROUNDS': get_env_int('ACTUARIAL_EARLY_STOPPING_ROUNDS', 50),
    
    # Ridge/Lasso settings
    'RIDGE_ALPHA': get_env_float('RIDGE_ALPHA', 1.0),
    'LASSO_ALPHA': get_env_float('LASSO_ALPHA', 0.1),
    
    # Random Forest settings
    'RANDOM_FOREST_N_ESTIMATORS': get_env_int('RANDOM_FOREST_N_ESTIMATORS', 200),
    'RANDOM_FOREST_MAX_DEPTH': get_env_int('RANDOM_FOREST_MAX_DEPTH', 10),
    
    # Hugging Face settings
    'UPLOAD_TO_HUB': get_env_bool('UPLOAD_TO_HUB', True),
    'HF_TOKEN': os.getenv('HF_TOKEN'),
    'HF_REPO_ID': os.getenv('HF_REPO_ID'),
}

print("=" * 60)
print("PIPELINE CONFIGURATION LOADED FROM .env")
print("=" * 60)
print(f"\nMLflow Settings:")
print(f"  CLEAN_MLFLOW_RUNS: {CONFIG['CLEAN_MLFLOW_RUNS']}")
print(f"  MLFLOW_EXPERIMENT_NAME: {CONFIG['MLFLOW_EXPERIMENT_NAME']}")
print(f"\nData Settings:")
print(f"  TEST_SIZE: {CONFIG['TEST_SIZE']}")
print(f"  RANDOM_STATE: {CONFIG['RANDOM_STATE']}")
print(f"\nHugging Face Settings:")
print(f"  UPLOAD_TO_HUB: {CONFIG['UPLOAD_TO_HUB']}")
print(f"  HF_REPO_ID: {CONFIG['HF_REPO_ID']}")
print("=" * 60)

Note: you may need to restart the kernel to use updated packages.
PIPELINE CONFIGURATION LOADED FROM .env

MLflow Settings:
  CLEAN_MLFLOW_RUNS: False
  MLFLOW_EXPERIMENT_NAME: Belgian_MTPL_Severity

Data Settings:
  TEST_SIZE: 0.2
  RANDOM_STATE: 42

Hugging Face Settings:
  UPLOAD_TO_HUB: True
  HF_REPO_ID: charlesnanakwakye/belgian-mtpl-claim-severity


In [3]:
display(df_beMTPL16.head())

,insurance_contract,policy_year,exposure,insured_birth_year,vehicle_age,policy_holder_age,driver_license_age,vehicle_brand,vehicle_model,mileage,vehicle_power,catalog_value,claim_value,number_of_liability_claims,number_of_bodily_injury_liability_claims,claim_time,claim_responsibility_rate,driving_training_label,signal
0,C1,1,0.386301,1945,10,9,40,MERCEDES,ME-1245,30000,75,983732,2,0,0,00:00,0,No,0
1,C2,1,0.493151,1941,4,25,24,VOLKSWAGEN,VO-2461,30000,55,510562,8,0,0,07:45,0,No,0
2,C3,1,0.290411,1944,0,2,39,AUDI,AU-967,30000,120,1934768,10,0,0,00:00,0,No,0
3,C4,1,0.336986,1948,1,14,37,LANCIA,LA-2346,30000,51,536755,13,0,0,18:50,0,No,0
4,C5,1,0.219178,1928,3,7,59,CITROEN,CI-1258,30000,54,446725,14,0,0,00:00,100,No,0


## 1. Data Validation & Understanding

### Complete Dataset Schema

| Column Name | Type | Description |

### 1.1 Data Inspection

- `info()` - Shows data types and non-null counts per column
- `describe()` - Statistical summaries for numeric columns (mean, min, max, quartiles)

In [4]:
print("Dataset Overview")
df_beMTPL16.info()

print("\nNull Values:")
print(df_beMTPL16.isnull().sum())

# Separate columns by type for analysis
cat_col = [col for col in df_beMTPL16.columns if df_beMTPL16[col].dtype == 'category']
num_col = [col for col in df_beMTPL16.columns if df_beMTPL16[col].dtype != 'category']

print(f'\nColumns: {len(cat_col)} categorical, {len(num_col)} numerical')

# Count zeros in numeric columns (potential missing values or data quality issues)
zero_counts = {col: (df_beMTPL16[col] == 0).sum() for col in num_col}
zero_df = pd.DataFrame([(col, count, f"{count/len(df_beMTPL16)*100:.2f}%") 
                         for col, count in zero_counts.items() if count > 0],
                       columns=['Column', 'Zero Count', 'Percentage'])
print("\nZero Values:")
display(zero_df.sort_values(by='Zero Count', ascending=False))

print("\nStatistics:")
display(df_beMTPL16.describe())

Dataset Overview
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70791 entries, 0 to 70790
Data columns (total 19 columns):
 #   Column                                    Non-Null Count  Dtype   
---  ------                                    --------------  -----   
 0   insurance_contract                        70791 non-null  category
 1   policy_year                               70791 non-null  int32   
 2   exposure                                  70791 non-null  float64 
 3   insured_birth_year                        70791 non-null  int32   
 4   vehicle_age                               70791 non-null  int32   
 5   policy_holder_age                         70791 non-null  int32   
 6   driver_license_age                        70791 non-null  int32   
 7   vehicle_brand                             70791 non-null  category
 8   vehicle_model                             70791 non-null  category
 9   mileage                                   70791 non-null  int32   
 10  vehic

,Column,Zero Count,Percentage
7,signal,70746,99.94%
5,number_of_bodily_injury_liability_claims,69381,98.01%
4,number_of_liability_claims,46080,65.09%
6,claim_responsibility_rate,36017,50.88%
3,catalog_value,21844,30.86%
0,vehicle_age,3304,4.67%
1,policy_holder_age,2331,3.29%
2,driver_license_age,3,0.00%



Statistics:


,policy_year,exposure,insured_birth_year,vehicle_age,policy_holder_age,driver_license_age,mileage,vehicle_power,catalog_value,claim_value,number_of_liability_claims,number_of_bodily_injury_liability_claims,claim_responsibility_rate,signal
count,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000,7.079100e+04,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000
mean,2.503934,0.437601,1941.394231,6.163849,9.651679,38.196875,28327.824158,77.962382,5.823853e+05,80780.786371,0.349070,0.019918,48.424362,0.000636
std,1.108830,0.180683,6.960452,4.803108,6.789322,9.549636,5711.127945,29.633158,5.460070e+05,45999.887220,0.476679,0.139719,49.628070,0.025205
min,1.000000,0.200000,1911.000000,0.000000,0.000000,0.000000,2500.000000,30.000000,0.000000e+00,2.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,0.287671,1936.000000,2.000000,4.000000,34.000000,30000.000000,55.000000,0.000000e+00,41491.500000,0.000000,0.000000,0.000000,0.000000
50%,3.000000,0.400000,1943.000000,5.000000,9.000000,41.000000,30000.000000,74.000000,5.506000e+05,82430.000000,0.000000,0.000000,0.000000,0.000000
75%,3.000000,0.556164,1947.000000,9.000000,14.000000,43.000000,30000.000000,92.000000,8.677665e+05,120472.000000,1.000000,0.000000,100.000000,0.000000
max,4.000000,1.000000,1952.000000,60.000000,30.000000,60.000000,30000.000000,487.000000,7.234528e+06,169694.000000,1.000000,1.000000,100.000000,1.000000


### 1.2 Data Cleaning Strategy

**Columns removed:**
- `insured_birth_year` - Redundant with `policy_holder_age`
- `driving_training_label` - Sparse, low predictive value
- `record_number` - Identifier only

**Zero value handling:**
- Rows with `claim_value = 0` removed (non-claims)
- Zeros in `policy_holder_age`, `driver_license_age`, `catalog_value` replaced with median

In [5]:
print("Data Cleaning")

# Columns to remove: redundant, identifiers, or low predictive value
columns_to_drop = [
    'insured_birth_year',       # Redundant with policy_holder_age
    'driving_training_label',   # Sparse indicator, low value
    'record_number'             # Just an identifier
]

df_model_data = df_beMTPL16.drop(columns=columns_to_drop, errors='ignore')

# Keep only actual claims (claim_value > 0)
df_model_data = df_model_data[df_model_data['claim_value'] > 0].copy()

print(f"Dataset after cleaning: {df_model_data.shape[0]} rows (removed {len(df_beMTPL16) - df_model_data.shape[0]} rows with zero/negative claims)")

# Replace zeros with median for columns where 0 is likely missing data
numeric_cols_with_zeros = ['policy_holder_age', 'driver_license_age', 'catalog_value']

for col in numeric_cols_with_zeros:
    if col in df_model_data.columns:
        zero_count = (df_model_data[col] == 0).sum()
        if zero_count > 0:
            # Use median of non-zero values to avoid skew from the zeros themselves
            median_value = df_model_data[df_model_data[col] > 0][col].median()
            df_model_data.loc[df_model_data[col] == 0, col] = median_value
            print(f"  {col}: Replaced {zero_count} zeros with median {median_value:.2f}")

print(f"\nFinal dataset: {df_model_data.shape[0]} rows, {df_model_data.shape[1]} columns")

Data Cleaning
Dataset after cleaning: 70791 rows (removed 0 rows with zero/negative claims)
  policy_holder_age: Replaced 2331 zeros with median 9.00
  driver_license_age: Replaced 3 zeros with median 41.00
  catalog_value: Replaced 21844 zeros with median 725094.00

Final dataset: 70791 rows, 17 columns


### 1.3 Feature Selection

**Target:** `claim_value` (EUR, log-transformed for training)

**Features (10):**
| Feature | Type | Notes |
| :--- | :--- | :--- |
| `policy_year` | Numeric | Year of policy |
| `vehicle_age` | Numeric | Age of vehicle |
| `policy_holder_age` | Numeric | Age of policyholder |
| `driver_license_age` | Numeric | Years since license |
| `vehicle_brand` | Categorical | Vehicle manufacturer |
| `vehicle_model` | Categorical | Vehicle model |
| `mileage` | Numeric | Vehicle mileage |
| `vehicle_power` | Numeric | Engine power |
| `catalog_value` | Numeric | Vehicle catalog value |
| `claim_time` | Categorical | Day/Night (transformed from timestamp) |

## 2. Feature Engineering & Data Preparation

- `claim_time` converted from raw timestamps to binary Day/Night (Night = 20:00-06:00)
- Target log-transformed via `log1p()` to handle right-skewed distribution
- 80/20 train/test split with fixed random seed for reproducibility

In [6]:
from sklearn.model_selection import train_test_split

print("Feature Engineering")

def time_to_day_night(time_str):
    """Convert time string (HH:MM) to Day/Night category.
    Night: 20:00-06:00, Day: 06:00-20:00
    """
    try:
        hour = int(str(time_str).split(':')[0])
        return 'Night' if (hour >= 20 or hour < 6) else 'Day'
    except:
        return 'Day'  # Default to Day for invalid values

# Transform claim_time before train/test split so model trains on Day/Night
df_model_data['claim_time'] = df_model_data['claim_time'].apply(time_to_day_night)
print(f"Converted claim_time to Day/Night: {df_model_data['claim_time'].value_counts().to_dict()}")

features = [
    'policy_year', 'vehicle_age', 'policy_holder_age', 'driver_license_age',
    'vehicle_brand', 'vehicle_model', 'mileage', 'vehicle_power',
    'catalog_value', 'claim_time'
]
target = 'claim_value'

X = df_model_data[features]
# Log transform target to handle right-skewed distribution (many small claims, few large)
y = np.log1p(df_model_data[target])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=CONFIG['TEST_SIZE'], 
    random_state=CONFIG['RANDOM_STATE']
)
print(f"Train: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")
print(f"  (test_size={CONFIG['TEST_SIZE']}, random_state={CONFIG['RANDOM_STATE']})")

Feature Engineering
Converted claim_time to Day/Night: {'Day': 39591, 'Night': 31200}
Train: 56632 rows, Test: 14159 rows
  (test_size=0.2, random_state=42)


## 3. Preprocessing Pipelines

**Standard Pipeline (Models 1-4, 6-7):**
- Numeric: StandardScaler
- Categorical: OneHotEncoder (min_frequency=0.01 to group rare categories)

**Actuarial Pipeline (Model 5):**
- Numeric: StandardScaler  
- High-cardinality (brand, model): TargetEncoder
- Low-cardinality (claim_time): OneHotEncoder

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("Preprocessing Pipeline")

numeric_features = [
    'policy_year', 'vehicle_age', 'policy_holder_age',
    'driver_license_age', 'mileage', 'vehicle_power', 'catalog_value'
]
categorical_features = ['vehicle_brand', 'vehicle_model', 'claim_time']

# Standard pipeline: scale numerics, one-hot encode categoricals
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[
    # min_frequency groups rare categories (< 1%) into "infrequent" to reduce feature explosion
    ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=0.01, sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

# Fit on training data only to avoid data leakage
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

processed_feature_names = preprocessor.get_feature_names_out()
print(f"Processed features: {X_train_processed.shape[1]}")

# --- Actuarial Pipeline ---
# Uses Target Encoding for high-cardinality features (brand, model)
# which encodes each category by its mean target value
print("\n\nActuarial XGBoost Preprocessing Pipeline (with Target Encoding)")
%pip install category_encoders -q
import category_encoders
from category_encoders import TargetEncoder

# Target Encoding needs original scale values (not log-transformed)
y_train_original = np.expm1(y_train)

numeric_transformer_actuarial = Pipeline(steps=[('scaler', StandardScaler())])
categorical_transformer_actuarial = ColumnTransformer(
    transformers=[
        # Target encode high-cardinality features (many unique values)
        ('target_high', TargetEncoder(), ['vehicle_brand', 'vehicle_model']),
        # OneHot for low-cardinality (Day/Night only has 2 values)
        ('onehot_low', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['claim_time'])
    ],
    remainder='drop'
)

preprocessor_actuarial = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_actuarial, numeric_features),
        ('cat', categorical_transformer_actuarial, categorical_features)
    ],
    remainder='drop'
)

# Fit with target values for Target Encoding
preprocessor_actuarial.fit(X_train, y_train_original)
X_train_actuarial = preprocessor_actuarial.transform(X_train)
X_test_actuarial = preprocessor_actuarial.transform(X_test)

print(f"Actuarial processed features: {X_train_actuarial.shape[1]}")
print("Target encoding applied to vehicle_brand and vehicle_model")

Preprocessing Pipeline
Processed features: 54


Actuarial XGBoost Preprocessing Pipeline (with Target Encoding)
Note: you may need to restart the kernel to use updated packages.
Actuarial processed features: 11
Target encoding applied to vehicle_brand and vehicle_model


## 4. Model Training

Seven models trained and compared:
1. **Linear Regression** - Baseline
2. **XGBoost (Basic)** - Gradient boosting with early stopping
3. **Random Forest** - Ensemble of 200 trees
4. **XGBoost (Tuned)** - Hyperparameter optimization via RandomizedSearchCV
5. **Actuarial XGBoost** - Gamma regression on original scale (Best for portfolio risk - lowest RMSE)
6. **Ridge Regression** - L2 regularization
7. **Lasso Regression** - L1 regularization with feature selection

**Metrics:** RMSE (EUR) for portfolio risk, MAPE for individual claim accuracy

**Model Selection:** Best model chosen by lowest MAPE (individual claim accuracy)

### 4.1 MLflow Setup

All models tracked via MLflow with autologging enabled. View dashboard: `mlflow ui` → http://localhost:5000

In [8]:
%pip install mlflow -q

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import os

# Set tracking URI (local file system)
mlruns_path = "./mlruns"
mlflow.set_tracking_uri(mlruns_path)

# Ensure mlruns directory and required subdirectories exist
if not os.path.exists(mlruns_path):
    os.makedirs(mlruns_path)
    print("Created ./mlruns directory")

# MLflow requires a .trash folder for deleted experiments
trash_path = os.path.join(mlruns_path, ".trash")
if not os.path.exists(trash_path):
    os.makedirs(trash_path)
    print("Created ./mlruns/.trash directory")

experiment_name = CONFIG['MLFLOW_EXPERIMENT_NAME']
try:
    experiment_id = mlflow.create_experiment(experiment_name)
except mlflow.exceptions.MlflowException:
    # Experiment already exists
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

mlflow.set_experiment(experiment_name)

mlflow.sklearn.autolog(log_input_examples=False, log_model_signatures=False)
mlflow.xgboost.autolog(log_input_examples=False, log_model_signatures=False)

print(f"MLflow configured successfully!")
print(f"Experiment: {experiment_name}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Run logs will be saved to ./mlruns directory")

Note: you may need to restart the kernel to use updated packages.


/Users/charlesnanakwakye/HobbyApps/be-insurance-ai/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)


MLflow configured successfully!
Experiment: Belgian_MTPL_Severity
Tracking URI: ./mlruns
Run logs will be saved to ./mlruns directory


### 4.2 Optional: Clean MLflow History

Set `CLEAN_MLFLOW_RUNS=True` in `.env` to delete all previous runs before training.

In [9]:
import shutil
import os

# Use configuration from .env file
CLEAN_MLFLOW_RUNS = CONFIG['CLEAN_MLFLOW_RUNS']

if CLEAN_MLFLOW_RUNS:
    mlruns_path = "./mlruns"
    if os.path.exists(mlruns_path):
        shutil.rmtree(mlruns_path)
        print("✓ Cleaned up old MLflow runs directory")
        print("  All previous experiment data has been deleted.")
        
        # Recreate the mlruns directory and .trash folder after cleanup
        os.makedirs(mlruns_path)
        os.makedirs(os.path.join(mlruns_path, ".trash"))
        print("✓ Recreated ./mlruns directory with .trash folder")
        
        # Re-initialize MLflow after cleanup
        mlflow.set_tracking_uri("./mlruns")
        experiment_name = CONFIG['MLFLOW_EXPERIMENT_NAME']
        experiment_id = mlflow.create_experiment(experiment_name)
        mlflow.set_experiment(experiment_name)
        print(f"✓ Re-initialized MLflow experiment: {experiment_name}")
    else:
        print("No MLflow runs directory found - nothing to clean")
else:
    print("✓ Cleanup skipped (CLEAN_MLFLOW_RUNS=False in .env)")
    print("  To clean up old runs, set CLEAN_MLFLOW_RUNS=True in .env file")

✓ Cleanup skipped (CLEAN_MLFLOW_RUNS=False in .env)
  To clean up old runs, set CLEAN_MLFLOW_RUNS=True in .env file


In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="Baseline_Linear_Regression"):
    model_lr = LinearRegression()
    model_lr.fit(X_train_processed, y_train)
    pred_lr = model_lr.predict(X_test_processed)
    test_rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))
    
    # Convert to original scale for business metrics
    y_test_original = np.expm1(y_test)
    pred_lr_original = np.expm1(pred_lr)
    rmse_lr_eur = np.sqrt(mean_squared_error(y_test_original, pred_lr_original))
    mape_lr = mean_absolute_percentage_error(y_test_original, pred_lr_original)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_lr_eur)
    mlflow.log_metric("mape", mape_lr)
    
    print(f"Linear Regression RMSE (log scale): {test_rmse_lr:.6f}")
    print(f"  RMSE (EUR): €{rmse_lr_eur:.2f}")
    print(f"  MAPE: {mape_lr * 100:.2f}%")

2025/11/29 04:59:43 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/11/29 04:59:45 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Linear Regression RMSE (log scale): 0.506865
  RMSE (EUR): €24614.96
  MAPE: 59.86%


### 4.3 Model 1: Linear Regression (Baseline)

In [11]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="XGBoost_Basic"):
    model_xgb = xgb.XGBRegressor(
        n_estimators=CONFIG['XGBOOST_N_ESTIMATORS'],
        learning_rate=CONFIG['XGBOOST_LEARNING_RATE'],
        max_depth=CONFIG['XGBOOST_MAX_DEPTH'],
        early_stopping_rounds=CONFIG['XGBOOST_EARLY_STOPPING_ROUNDS'],
        random_state=CONFIG['RANDOM_STATE']
    )
    model_xgb.fit(
        X_train_processed, y_train,
        eval_set=[(X_test_processed, y_test)],
        verbose=False
    )
    pred_xgb = model_xgb.predict(X_test_processed)
    test_rmse_xgb = np.sqrt(mean_squared_error(y_test, pred_xgb))
    
    # Convert to original scale for business metrics
    y_test_original = np.expm1(y_test)
    pred_xgb_original = np.expm1(pred_xgb)
    rmse_xgb_eur = np.sqrt(mean_squared_error(y_test_original, pred_xgb_original))
    mape_xgb = mean_absolute_percentage_error(y_test_original, pred_xgb_original)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_xgb_eur)
    mlflow.log_metric("mape", mape_xgb)
    
    print(f"XGBoost (Basic) RMSE (log scale): {test_rmse_xgb:.6f}")
    print(f"  RMSE (EUR): €{rmse_xgb_eur:.2f}")
    print(f"  MAPE: {mape_xgb * 100:.2f}%")

2025/11/29 04:59:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/29 04:59:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


XGBoost (Basic) RMSE (log scale): 0.430417
  RMSE (EUR): €12773.95
  MAPE: 40.53%


### 4.4 Model 2: XGBoost (Basic)

In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="RF_Ensemble"):
    model_rf = RandomForestRegressor(
        n_estimators=CONFIG['RANDOM_FOREST_N_ESTIMATORS'],
        max_depth=CONFIG['RANDOM_FOREST_MAX_DEPTH'],
        random_state=CONFIG['RANDOM_STATE'],
        n_jobs=-1
    )
    model_rf.fit(X_train_processed, y_train)
    pred_rf = model_rf.predict(X_test_processed)
    test_rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))
    
    # Convert from log scale back to EUR (expm1 reverses log1p transformation)
    y_test_original = np.expm1(y_test)
    pred_rf_original = np.expm1(pred_rf)
    rmse_rf_eur = np.sqrt(mean_squared_error(y_test_original, pred_rf_original))
    mape_rf = mean_absolute_percentage_error(y_test_original, pred_rf_original)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_rf_eur)
    mlflow.log_metric("mape", mape_rf)
    
    print(f"Random Forest RMSE (log scale): {test_rmse_rf:.6f}")
    print(f"  RMSE (EUR): €{rmse_rf_eur:.2f}")
    print(f"  MAPE: {mape_rf * 100:.2f}%")

2025/11/29 04:59:47 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/11/29 04:59:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Random Forest RMSE (log scale): 0.431980
  RMSE (EUR): €12801.94
  MAPE: 40.53%


### 4.5 Model 3: Random Forest

In [13]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="XGB_Tuned_RandomSearch"):
    # Define hyperparameter search space
    param_dist = {
        'n_estimators': randint(200, 1500),      # Number of trees
        'learning_rate': uniform(0.01, 0.1),      # Step size shrinkage
        'max_depth': randint(3, 8),               # Tree depth (complexity)
        'subsample': uniform(0.7, 0.3),           # Row sampling per tree
        'colsample_bytree': uniform(0.7, 0.3)     # Column sampling per tree
    }

    # RandomizedSearchCV: random sampling from param space is faster than grid search
    random_search = RandomizedSearchCV(
        xgb.XGBRegressor(random_state=CONFIG['RANDOM_STATE']),
        param_distributions=param_dist,
        n_iter=CONFIG['RANDOM_SEARCH_N_ITER'],
        cv=CONFIG['RANDOM_SEARCH_CV_FOLDS'],
        scoring='neg_root_mean_squared_error',
        n_jobs=-1,  # Use all CPU cores
        random_state=CONFIG['RANDOM_STATE'],
        verbose=0
    )
    random_search.fit(X_train_processed, y_train)
    
    best_model_xgb = random_search.best_estimator_
    log_predictions_tuned = best_model_xgb.predict(X_test_processed)
    final_test_rmse = np.sqrt(mean_squared_error(y_test, log_predictions_tuned))
    
    # Convert from log scale back to EUR for business metrics
    y_test_original = np.expm1(y_test)
    predictions_original = np.expm1(log_predictions_tuned)
    rmse_tuned_eur = np.sqrt(mean_squared_error(y_test_original, predictions_original))
    mape_tuned = mean_absolute_percentage_error(y_test_original, predictions_original)
    
    mlflow.log_params(random_search.best_params_)
    mlflow.log_metric("rmse_eur", rmse_tuned_eur)
    mlflow.log_metric("mape", mape_tuned)

    print(f"XGBoost (Tuned) RMSE (log scale): {final_test_rmse:.6f}")
    print(f"  RMSE (EUR): €{rmse_tuned_eur:.2f}")
    print(f"  MAPE: {mape_tuned * 100:.2f}%")
    print(f"\nBest Hyperparameters:")
    for param, value in random_search.best_params_.items():
        print(f"  {param}: {value}")

2025/11/29 04:59:54 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/11/29 05:02:51 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/29 05:02:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/29 05:02:53 INFO mlflow.sklearn.utils: Logging the 5 best runs, 45 runs will be omitted.
2025/11/29 05:02:53 WARNING mlflow.sklearn: Encountered exception during creation of child runs for parameter search. Child runs may be missing. Exception: 'NoneType' object has no attribute '_to_mlflow_entity'


XGBoost (Tuned) RMSE (log scale): 0.430198
  RMSE (EUR): €12816.87
  MAPE: 40.82%

Best Hyperparameters:
  colsample_bytree: 0.8023199053150775
  learning_rate: 0.02134735212405891
  max_depth: 4
  n_estimators: 462
  subsample: 0.8979952138102536


### 4.6 Model 4: XGBoost (Tuned)

Hyperparameters optimized via RandomizedSearchCV (50 iterations, 5-fold CV).

### 4.7 Models 5-7: Ridge, Lasso, Actuarial XGBoost

In [14]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="Ridge_Regression"):
    model_ridge = Ridge(alpha=CONFIG['RIDGE_ALPHA'], random_state=CONFIG['RANDOM_STATE'])
    model_ridge.fit(X_train_processed, y_train)
    pred_ridge = model_ridge.predict(X_test_processed)
    test_rmse_ridge = np.sqrt(mean_squared_error(y_test, pred_ridge))
    
    # Convert to original scale for business metrics
    y_test_original = np.expm1(y_test)
    pred_ridge_original = np.expm1(pred_ridge)
    rmse_ridge_eur = np.sqrt(mean_squared_error(y_test_original, pred_ridge_original))
    mape_ridge = mean_absolute_percentage_error(y_test_original, pred_ridge_original)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_ridge_eur)
    mlflow.log_metric("mape", mape_ridge)
    
    print(f"Ridge Regression RMSE (log scale): {test_rmse_ridge:.6f}")
    print(f"  RMSE (EUR): €{rmse_ridge_eur:.2f}")
    print(f"  MAPE: {mape_ridge * 100:.2f}%")

2025/11/29 05:02:53 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/11/29 05:02:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Ridge Regression RMSE (log scale): 0.506865
  RMSE (EUR): €24613.65
  MAPE: 59.86%


### 4.8 Model 6: Lasso Regression

L1 regularization can zero out coefficients, providing automatic feature selection.

In [15]:
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="Lasso_Regression"):
    model_lasso = Lasso(alpha=CONFIG['LASSO_ALPHA'], random_state=CONFIG['RANDOM_STATE'], max_iter=5000)
    model_lasso.fit(X_train_processed, y_train)
    pred_lasso = model_lasso.predict(X_test_processed)
    test_rmse_lasso = np.sqrt(mean_squared_error(y_test, pred_lasso))
    
    # Convert to original scale for business metrics
    y_test_original = np.expm1(y_test)
    pred_lasso_original = np.expm1(pred_lasso)
    rmse_lasso_eur = np.sqrt(mean_squared_error(y_test_original, pred_lasso_original))
    mape_lasso = mean_absolute_percentage_error(y_test_original, pred_lasso_original)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_lasso_eur)
    mlflow.log_metric("mape", mape_lasso)
    
    # Show feature selection (how many features were zeroed out)
    non_zero_features = (model_lasso.coef_ != 0).sum()
    total_features = len(model_lasso.coef_)
    
    print(f"Lasso Regression RMSE (log scale): {test_rmse_lasso:.6f}")
    print(f"  RMSE (EUR): €{rmse_lasso_eur:.2f}")
    print(f"  MAPE: {mape_lasso * 100:.2f}%")
    print(f"  Feature Selection: {non_zero_features}/{total_features} features kept (removed {total_features - non_zero_features})")

2025/11/29 05:02:54 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/11/29 05:02:56 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Lasso Regression RMSE (log scale): 0.518071
  RMSE (EUR): €19098.52
  MAPE: 63.02%
  Feature Selection: 1/54 features kept (removed 53)


In [16]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="Actuarial_XGB_Gamma"):
    # Gamma regression requires original scale values (not log-transformed)
    # because Gamma distribution models right-skewed positive values directly
    y_train_original = np.expm1(y_train)
    y_test_original = np.expm1(y_test)

    # objective='reg:gamma' uses Gamma deviance loss, designed for insurance severity
    model_xgb_actuarial = xgb.XGBRegressor(
        objective='reg:gamma',
        n_estimators=CONFIG['ACTUARIAL_N_ESTIMATORS'],
        learning_rate=CONFIG['ACTUARIAL_LEARNING_RATE'],
        max_depth=CONFIG['ACTUARIAL_MAX_DEPTH'],
        subsample=CONFIG['ACTUARIAL_SUBSAMPLE'],
        colsample_bytree=CONFIG['ACTUARIAL_COLSAMPLE_BYTREE'],
        early_stopping_rounds=CONFIG['ACTUARIAL_EARLY_STOPPING_ROUNDS'],
        random_state=CONFIG['RANDOM_STATE'],
        n_jobs=-1
    )

    # Uses actuarial preprocessor (with Target Encoding instead of OneHot)
    model_xgb_actuarial.fit(
        X_train_actuarial, y_train_original,
        eval_set=[(X_test_actuarial, y_test_original)],
        verbose=False
    )

    pred_xgb_actuarial = model_xgb_actuarial.predict(X_test_actuarial)

    # Already on original scale, no conversion needed
    rmse_actuarial = np.sqrt(mean_squared_error(y_test_original, pred_xgb_actuarial))
    mape_actuarial = mean_absolute_percentage_error(y_test_original, pred_xgb_actuarial)
    
    mlflow.log_metric("rmse_eur", rmse_actuarial)
    mlflow.log_metric("mape", mape_actuarial)

    print(f"Actuarial XGBoost (Gamma Regression) Performance:")
    print(f"  RMSE (original scale, EUR): €{rmse_actuarial:.2f}")
    print(f"  MAPE: {mape_actuarial * 100:.2f}%")

2025/11/29 05:02:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/29 05:02:58 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Actuarial XGBoost (Gamma Regression) Performance:
  RMSE (original scale, EUR): €12719.62
  MAPE: 49.99%


### 4.9 Model 7: Actuarial XGBoost (Gamma Regression)

Uses `objective='reg:gamma'` trained on original (non-log) claim values. Designed for right-skewed insurance data.

In [17]:
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd

# Convert test set back to EUR for comparison
y_test_original = np.expm1(y_test)

# Config dict with lambdas for lazy evaluation (handles missing models gracefully)
model_configs = {
    'Linear Regression': {
        'rmse': lambda: np.sqrt(mean_squared_error(y_test_original, np.expm1(pred_lr))),
        'mape': lambda: mean_absolute_percentage_error(y_test_original, np.expm1(pred_lr)),
        'pred': lambda: np.expm1(pred_lr)
    },
    'XGBoost (basic)': {
        'rmse': lambda: np.sqrt(mean_squared_error(y_test_original, np.expm1(pred_xgb))),
        'mape': lambda: mean_absolute_percentage_error(y_test_original, np.expm1(pred_xgb)),
        'pred': lambda: np.expm1(pred_xgb)
    },
    'Random Forest': {
        'rmse': lambda: np.sqrt(mean_squared_error(y_test_original, np.expm1(pred_rf))),
        'mape': lambda: mean_absolute_percentage_error(y_test_original, np.expm1(pred_rf)),
        'pred': lambda: np.expm1(pred_rf)
    },
    'XGBoost (tuned)': {
        'rmse': lambda: rmse_tuned_eur,
        'mape': lambda: mape_tuned,
        'pred': lambda: np.expm1(log_predictions_tuned)
    },
    'Actuarial XGBoost': {
        'rmse': lambda: rmse_actuarial,
        'mape': lambda: mape_actuarial,
        'pred': lambda: pred_xgb_actuarial  # Already on original scale
    },
    'Ridge Regression': {
        'rmse': lambda: rmse_ridge_eur,
        'mape': lambda: mape_ridge,
        'pred': lambda: pred_ridge_original
    },
    'Lasso Regression': {
        'rmse': lambda: rmse_lasso_eur,
        'mape': lambda: mape_lasso,
        'pred': lambda: pred_lasso_original
    }
}

# Collect metrics from all trained models
models_rmse = {}
models_mape = {}
models_pred = {}

for name, config in model_configs.items():
    try:
        models_rmse[name] = config['rmse']()
        models_mape[name] = config['mape']()
        models_pred[name] = config['pred']()
    except NameError:
        pass  # Skip models that weren't trained

model_list = list(models_rmse.keys())

# --- Print Results ---
print("=" * 80)
print(f"MODEL COMPARISON: {len(model_list)} MODELS")
print("=" * 80)
print("\nNote: Detailed metrics, hyperparameters, and model artifacts are now")
print("tracked in MLflow. View the dashboard with: mlflow ui")
print("=" * 80)

print("\nModel Performance (RMSE on original scale, EUR):")
for i, name in enumerate(model_list, 1):
    print(f"  {i}. {name:<30} €{models_rmse[name]:.2f}")

print("\nModel Performance (MAPE - lower is better for individual claims):")
for i, name in enumerate(model_list, 1):
    print(f"  {i}. {name:<30} {models_mape[name] * 100:.2f}%")

# Find best model by lowest MAPE (best for individual claim accuracy)
# Note: RMSE is better for portfolio risk, MAPE is better for individual claims
print("\n" + "=" * 80)
print("BEST MODEL SUMMARY (Selected by lowest MAPE for individual claim accuracy)")
print("=" * 80)

best_model = min(models_mape, key=models_mape.get)
print(f"\nBest Model: {best_model}")
print(f"  RMSE: €{models_rmse[best_model]:.2f}")
print(f"  MAPE: {models_mape[best_model] * 100:.2f}%")
print(f"  Average Claim Value: €{y_test_original.mean():.2f}")
print(f"\nNote: This model was selected for best individual claim accuracy (lowest MAPE).")
print(f"      For portfolio/reserve calculations, Actuarial XGBoost (lowest RMSE) may be preferred.")

# Show sample predictions
print("\n" + "=" * 80)
print("DETAILED PREDICTIONS ON TEST SET (First 20 claims)")
print("=" * 80)

results_dict = {'Actual': y_test_original}
results_dict.update({name: models_pred[name] for name in model_list})
results_df = pd.DataFrame(results_dict).reset_index(drop=True)

print(f"\nShowing actual claim values vs predictions from all {len(model_list)} models (in EUR):\n")
display(results_df.head(20).round(2))

# Show prediction errors
print("\n" + "=" * 80)
print("MODEL PREDICTION ERRORS (First 20 claims)")
print("=" * 80)
print("\nAbsolute Error (EUR) for each model:\n")

errors_dict = {'Actual': y_test_original}
errors_dict.update({f'{name} Error': np.abs(models_pred[name] - y_test_original) for name in model_list})
errors_df = pd.DataFrame(errors_dict).reset_index(drop=True)
display(errors_df.head(20).round(2))

# Error statistics across all test samples
print("\n" + "=" * 80)
print("MODEL ERROR STATISTICS")
print("=" * 80)

error_stats = pd.DataFrame({
    'Model': model_list,
    'Mean Absolute Error (EUR)': [np.abs(models_pred[name] - y_test_original).mean() for name in model_list],
    'Median Absolute Error (EUR)': [np.median(np.abs(models_pred[name] - y_test_original)) for name in model_list],
    'Max Error (EUR)': [np.abs(models_pred[name] - y_test_original).max() for name in model_list]
})

print("\n")
display(error_stats.round(2))

print("\n" + "=" * 80)
print("TO VIEW MLFLOW DASHBOARD:")
print("   Run in terminal: mlflow ui")
print("   Then open: http://localhost:5000")
print("=" * 80)

MODEL COMPARISON: 7 MODELS

Note: Detailed metrics, hyperparameters, and model artifacts are now
tracked in MLflow. View the dashboard with: mlflow ui

Model Performance (RMSE on original scale, EUR):
  1. Linear Regression              €24614.96
  2. XGBoost (basic)                €12773.95
  3. Random Forest                  €12801.94
  4. XGBoost (tuned)                €12816.87
  5. Actuarial XGBoost              €12719.62
  6. Ridge Regression               €24613.65
  7. Lasso Regression               €19098.52

Model Performance (MAPE - lower is better for individual claims):
  1. Linear Regression              59.86%
  2. XGBoost (basic)                40.53%
  3. Random Forest                  40.53%
  4. XGBoost (tuned)                40.82%
  5. Actuarial XGBoost              49.99%
  6. Ridge Regression               59.86%
  7. Lasso Regression               63.02%

BEST MODEL SUMMARY (Selected by lowest MAPE for individual claim accuracy)

Best Model: Random Forest
  RMSE

,Actual,Linear Regression,XGBoost (basic),Random Forest,XGBoost (tuned),Actuarial XGBoost,Ridge Regression,Lasso Regression
0,5070.0,21127.05,12972.959961,14158.41,12943.410156,18594.400391,21127.08,23950.34
1,37927.0,22000.58,18154.820312,15903.04,18165.750000,22295.289062,22000.04,23950.34
2,122110.0,158663.66,137687.140625,137850.64,137657.203125,139281.000000,158661.10,152449.80
3,86971.0,90146.74,103469.312500,105424.39,101603.773438,102539.953125,90151.54,82260.46
4,129777.0,166740.13,135250.437500,137062.76,136294.078125,130157.148438,166740.94,152449.80
5,35177.0,22147.77,18110.740234,16077.66,16740.089844,20656.759766,22146.38,23950.34
6,142096.0,173826.77,138687.390625,138856.28,135396.921875,134946.609375,173828.46,152449.80
7,136164.0,163316.10,136409.156250,134903.83,135241.031250,132602.046875,163313.46,152449.80
8,33640.0,20686.09,15057.540039,15267.10,15362.950195,24402.130859,20685.98,23950.34
9,27857.0,21608.73,17976.460938,16808.12,17565.189453,23039.330078,21609.24,23950.34



MODEL PREDICTION ERRORS (First 20 claims)

Absolute Error (EUR) for each model:



,Actual,Linear Regression Error,XGBoost (basic) Error,Random Forest Error,XGBoost (tuned) Error,Actuarial XGBoost Error,Ridge Regression Error,Lasso Regression Error
0,5070.0,16057.05,7902.96,9088.41,7873.41,13524.40,16057.08,18880.34
1,37927.0,15926.42,19772.18,22023.96,19761.25,15631.71,15926.96,13976.66
2,122110.0,36553.66,15577.14,15740.64,15547.20,17171.00,36551.10,30339.80
3,86971.0,3175.74,16498.31,18453.39,14632.77,15568.95,3180.54,4710.54
4,129777.0,36963.13,5473.44,7285.76,6517.08,380.15,36963.94,22672.80
5,35177.0,13029.23,17066.26,19099.34,18436.91,14520.24,13030.62,11226.66
6,142096.0,31730.77,3408.61,3239.72,6699.08,7149.39,31732.46,10353.80
7,136164.0,27152.10,245.16,1260.17,922.97,3561.95,27149.46,16285.80
8,33640.0,12953.91,18582.46,18372.90,18277.05,9237.87,12954.02,9689.66
9,27857.0,6248.27,9880.54,11048.88,10291.81,4817.67,6247.76,3906.66



MODEL ERROR STATISTICS




,Model,Mean Absolute Error (EUR),Median Absolute Error (EUR),Max Error (EUR)
0,Linear Regression,19806.10,17604.55,140085.50
1,XGBoost (basic),10003.77,9493.75,147654.96
2,Random Forest,10001.60,9551.20,150702.37
3,XGBoost (tuned),10050.29,9579.78,147146.13
4,Actuarial XGBoost,10001.65,9581.56,145238.87
5,Ridge Regression,19805.29,17605.51,140084.75
6,Lasso Regression,15148.41,13650.66,136007.66



TO VIEW MLFLOW DASHBOARD:
   Run in terminal: mlflow ui
   Then open: http://localhost:5000


## 5. Results Analysis

Compares all models by RMSE (EUR) and MAPE. Shows predictions vs actuals and error statistics.

## 6. Model Artifacts

**Storage locations:**
- MLflow: `./mlruns/` (all models with full metadata, hyperparameters, metrics)
- Hugging Face Hub: Cloud storage for deployment

In [18]:
import joblib
import os

print("=" * 80)
print("BEST MODEL SUMMARY")
print("=" * 80)

print(f"\nBest Model: {best_model}")
print(f"  RMSE (EUR): €{models_rmse[best_model]:.2f}")
print(f"  MAPE: {models_mape[best_model] * 100:.2f}%")
print(f"  Average Claim Value: €{y_test_original.mean():.2f}")

# Map model names to their trained objects
model_mapping = {
    'Linear Regression': lambda: model_lr,
    'XGBoost (basic)': lambda: model_xgb,
    'Random Forest': lambda: model_rf,
    'XGBoost (tuned)': lambda: best_model_xgb,
    'Actuarial XGBoost': lambda: model_xgb_actuarial,
    'Ridge Regression': lambda: model_ridge,
    'Lasso Regression': lambda: model_lasso
}

# Collect all trained models (skip any that failed)
model_objects = {}
for name, get_model in model_mapping.items():
    try:
        model_objects[name] = get_model()
    except NameError:
        pass

if best_model not in model_objects:
    print(f"\nWarning: Best model '{best_model}' not found.")
    print("   Please run the corresponding training cell first.")
else:
    best_model_object = model_objects[best_model]
    print(f"\nFeatures used: {len(features)}")
    for i, feature in enumerate(features, 1):
        print(f"  {i}. {feature}")

print("\n" + "=" * 80)
print("Model artifacts tracked in MLflow. View with: mlflow ui")
print("=" * 80)

BEST MODEL SUMMARY

Best Model: Random Forest
  RMSE (EUR): €12801.94
  MAPE: 40.53%
  Average Claim Value: €80685.36

Features used: 10
  1. policy_year
  2. vehicle_age
  3. policy_holder_age
  4. driver_license_age
  5. vehicle_brand
  6. vehicle_model
  7. mileage
  8. vehicle_power
  9. catalog_value
  10. claim_time

Model artifacts tracked in MLflow. View with: mlflow ui


## 7. Deploy to Hugging Face Hub (Optional)

Set `UPLOAD_TO_HUB=True` and `HF_TOKEN` in `.env` to upload best model to Hugging Face for cloud access.

**Files uploaded:** `model.pkl`, `preprocessor.pkl`, `categories.json`, `metadata.json`

In [19]:
import os
import json
import joblib
from datetime import datetime

# Configuration from .env (already loaded via CONFIG)
UPLOAD_TO_HUB = CONFIG['UPLOAD_TO_HUB']
HF_TOKEN = CONFIG['HF_TOKEN']
HF_REPO_ID = CONFIG['HF_REPO_ID']

print(f"Configuration loaded from .env file:")
print(f"  UPLOAD_TO_HUB = {UPLOAD_TO_HUB}")
print(f"  Repository ID: {HF_REPO_ID}")

Configuration loaded from .env file:
  UPLOAD_TO_HUB = True
  Repository ID: charlesnanakwakye/belgian-mtpl-claim-severity


### 7.1 Upload to Hub

Uploads model, preprocessor, categories, and metadata to Hugging Face repository.

In [20]:
from huggingface_hub import HfApi, login
import shutil

if not UPLOAD_TO_HUB:
    print("Upload disabled")
else:
    login(token=HF_TOKEN)
    
    temp_dir = "./hf_upload_temp"
    os.makedirs(temp_dir, exist_ok=True)
    
    # Map model names to trained objects
    model_map = {
        'Linear Regression': model_lr,
        'XGBoost (basic)': model_xgb,
        'Random Forest': model_rf,
        'XGBoost (tuned)': best_model_xgb,
        'Actuarial XGBoost': model_xgb_actuarial,
        'Ridge Regression': model_ridge,
        'Lasso Regression': model_lasso
    }
    
    selected_model = model_map[best_model]
    # Actuarial model uses different preprocessor (Target Encoding vs OneHot)
    selected_preprocessor = preprocessor_actuarial if best_model == 'Actuarial XGBoost' else preprocessor
    
    # Serialize model and preprocessor
    joblib.dump(selected_model, os.path.join(temp_dir, "model.pkl"))
    joblib.dump(selected_preprocessor, os.path.join(temp_dir, "preprocessor.pkl"))
    
    # Save valid categories for each categorical feature
    # This allows the Streamlit app to populate dropdowns with valid options
    categories = {
        'vehicle_brand': sorted(X_train['vehicle_brand'].dropna().unique().tolist()),
        'vehicle_model': sorted(X_train['vehicle_model'].dropna().unique().tolist()),
        'claim_time': ['Day', 'Night']  # Binary after Day/Night transformation
    }
    with open(os.path.join(temp_dir, "categories.json"), 'w') as f:
        json.dump(categories, f, indent=2)
    print(f"Saved categories: {len(categories['vehicle_brand'])} brands, {len(categories['vehicle_model'])} models, claim_time: {categories['claim_time']}")
    
    # Save model metadata for documentation
    with open(os.path.join(temp_dir, "metadata.json"), 'w') as f:
        json.dump({
            'model_name': best_model,
            'rmse_eur': float(models_rmse[best_model]),
            'mape': float(models_mape[best_model]),
            'features': features,
            'uploaded_date': datetime.now().isoformat()
        }, f, indent=2)
    
    # Upload to Hugging Face Hub
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_ID, private=False, exist_ok=True)
    api.upload_folder(folder_path=temp_dir, repo_id=HF_REPO_ID, commit_message=f"Upload {best_model} with categories")
    
    # Clean up temp directory
    shutil.rmtree(temp_dir)
    print(f"Uploaded to: https://huggingface.co/{HF_REPO_ID}")

/Users/charlesnanakwakye/HobbyApps/be-insurance-ai/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Saved categories: 63 brands, 982 models, claim_time: ['Day', 'Night']


Processing Files (2 / 2): 100%|██████████| 18.0MB / 18.0MB, 39.3kB/s  
New Data Upload: 100%|██████████| 55.0kB / 55.0kB, 39.3kB/s  


Uploaded to: https://huggingface.co/charlesnanakwakye/belgian-mtpl-claim-severity


## Summary

**Pipeline completed:**
1. Loaded 70,791 claims from beMTPL16 dataset (`./data/`)
2. Cleaned data, engineered 10 features (including Day/Night transformation)
3. Trained 7 models with MLflow tracking
4. Best model (selected by lowest MAPE for individual claims) uploaded to Hugging Face Hub

**Model Selection:**
- **Random Forest** selected for best individual claim accuracy (lowest MAPE)
- For portfolio risk calculations, Actuarial XGBoost (lowest RMSE) is also available

**Key locations:**
- `./data/` - Cached dataset (auto-downloaded)
- `./mlruns/` - MLflow experiment history
- Hugging Face Hub - Production model deployment

**Next steps:**
- Run `mlflow ui` to view experiment dashboard
- Build Streamlit app using model from Hugging Face Hub